# 🧠 End-to-End Fine-Tuning & Evaluation Pipeline for Fraud Detection (W&B Lineage + Full Metrics)

## 📋 Project Overview & Prerequisites

This notebook implements the complete end-to-end pipeline for Supervised Fine-Tuning (SFT) and subsequent evaluation of **meta-llama/Llama-3.1-8B-Instruct** for detecting fraudulent phone calls (Fraud Detection) — fully integrated into **Weights & Biases (W&B) Artifact, Training Curve & Lineage Tracking** (`nemo-fraud-detection`).

### ⚙️ System Requirements

* **Docker** with the NVIDIA Container Toolkit (`nemo-automodel` container).
* An **NVIDIA NGC API Key** as well as a **Weights & Biases API Key** (`WANDB_API_KEY`).
* An **NVIDIA GPU** with at least 20 GB VRAM (Note: Run `docker compose stop nim-llama3` before training so that GPU memory is free).

### 📊 Estimated VRAM Requirements by Configuration

| Configuration | Model | Method | Estimated VRAM |
| :--- | :--- | :--- | :--- |
| `qwen2_1p5b_sft.yaml` | Qwen2-1.5B-Instruct | Full SFT | ~20 GB |
| `llama3p2_3b_sft.yaml` | Llama-3.2-3B-Instruct | Full SFT | ~38 GB |
| `llama3p1_8b_lora.yaml` | Llama-3.1-8B-Instruct | LoRA | ~24 GB |

---

### 🔗 W&B End-to-End Lineage & Metrics Overview

1. **Environment & GPU Check:** Verification of NVIDIA driver (`nvidia-smi`), CUDA availability, W&B login, and `automodel` CLI.
2. **Data Preparation (`stage-2-data-prep`):** Loads `curated-sft-dataset:latest` from W&B (with local fallback), formats the transcripts into the prompt-response schema, and logs `prepared-sft-dataset`.
3. **Training (`stage-3-finetuning`):** Executes `automodel` with W&B training telemetry enabled (`WANDB_RUN_ID` linked so `train/loss`, `val/loss`, `lr`, `grad_norm`, and throughput stream directly into the same run) and registers checkpoints as `llama3p1-8b-fraud-lora`.
4. **Comprehensive Evaluation (`stage-4-evaluation`):** Compares the base model (zero-shot) and fine-tuned model on the test set using both greedy generation and first-token log-probabilities (`fraud` vs. `legitimate`), logging:
   - **Classification & Imbalance Metrics:** Accuracy, Precision, Recall, F1 (for both `fraud` and `legitimate`), False Positive Rate (FPR), False Negative Rate (FNR), Matthews Correlation Coefficient (MCC), and Cohen's Kappa.
   - **Probability & Calibration Metrics:** ROC-AUC, Precision-Recall AUC (PR-AUC), Brier Score, ROC Curve, and Precision-Recall Curve.
   - **Error Analysis & Operational Metrics:** Confusion Matrix, Unparsed Output Count, Average Latency, `model_comparison` table, `prediction_samples` table, and a filtered `misclassified_samples` error-analysis table.

In [ ]:
# Optional: If the /opt/venv kernel is not registered yet, run once and switch kernel
import sys
import subprocess

print("Python executable:", sys.executable)
res = subprocess.run([sys.executable, "-m", "pip", "list"], capture_output=True, text=True)
for line in res.stdout.splitlines():
    if any(pkg in line.lower() for pkg in ["torch", "transformers", "peft", "automodel", "wandb"]):
        print(line)

In [ ]:
# Import all required libraries & central W&B configuration
import json
import os
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import torch
import wandb
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_auc_score,
)
from transformers import AutoModelForCausalLM, AutoTokenizer

# Central W&B project & artifact names (End-to-End Lineage)
WANDB_PROJECT = "nemo-fraud-detection"
CURATED_DATASET_ARTIFACT = "curated-sft-dataset:latest"
PREPARED_DATASET_ARTIFACT = "prepared-sft-dataset"
MODEL_ARTIFACT_NAME = "llama3p1-8b-fraud-lora"
EVAL_REPORT_ARTIFACT = "post-training-evaluation-report"

print("✅ Libraries and W&B configuration loaded successfully.")

### 1. Environment, GPU & W&B Check

Ensures that the GPU is visible, sufficient free VRAM is available, the `automodel` CLI is ready, and W&B is authenticated.

In [ ]:
def check_environment():
    print("=== Checking Environment, GPU, and W&B (Automodel Container) ===")

    # nvidia-smi check (including free memory to detect conflicts e.g. with nim-llama3)
    try:
        smi_output = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,memory.free,driver_version",
                "--format=csv",
            ],
            capture_output=True,
            text=True,
            check=True,
        ).stdout
        print(smi_output)
    except Exception as e:
        print(f"Warning running nvidia-smi: {e}")

    # PyTorch CUDA check
    print("CUDA available:", torch.cuda.is_available())
    assert torch.cuda.is_available(), (
        "No GPU visible — ensure the container was started with --gpus all."
    )
    print("Device:", torch.cuda.get_device_name(0))
    free_mem, total_mem = torch.cuda.mem_get_info(0)
    print(f"Free / total VRAM: {free_mem / 1e9:.1f} GB / {total_mem / 1e9:.1f} GB")
    if free_mem / 1e9 < 18.0:
        print(
            "⚠️ WARNING: Less than 18 GB of free VRAM! "
            "If nim-llama3 is still running, stop it before training with: docker compose stop nim-llama3"
        )

    # automodel CLI check
    automodel_path = shutil.which("automodel")
    assert automodel_path, (
        "automodel CLI not found — please select the 'Python (/opt/venv)' kernel in the Automodel container!"
    )
    print("automodel CLI found at:", automodel_path)

    # W&B login check
    wandb.login()
    print(f"W&B authenticated for project: {WANDB_PROJECT}")


check_environment()

### 2. Data Preparation (with W&B Artifact Lineage)

Fetches `curated-sft-dataset:latest` directly from W&B (falling back to the local repository path if unavailable), applies the unified prompt template, saves `train.jsonl`, `test.jsonl`, and `validation.jsonl` into the `data/` directory, and registers them as `prepared-sft-dataset` in W&B.

In [ ]:
PROMPT_TEMPLATE = (
    "Phone call transcript: {text}\n\n"
    "Question: Based on the phone call transcript above, is it a fraudulent call or a legitimate call?\n\n"
    "Answer: "
)


def prepare_data(run=None):
    own_run = False
    if run is None:
        run = wandb.init(
            project=WANDB_PROJECT,
            job_type="data-preparation",
            name="prepare-sft-prompt-dataset",
            tags=["stage-2-data-prep", "automodel"],
        )
        own_run = True

    print("\n=== Preparing Data ===")
    output_dir = Path.cwd() / "data"
    output_dir.mkdir(parents=True, exist_ok=True)

    # Use upstream artifact from W&B if available; otherwise fall back to local repo path
    try:
        upstream_art = run.use_artifact(CURATED_DATASET_ARTIFACT)
        source_dir = Path(upstream_art.download())
        print(f"Using W&B artifact: {CURATED_DATASET_ARTIFACT}")
    except Exception as exc:
        repo_root = Path("/data/nemo-fraud-detection-notebooks")
        source_dir = repo_root / "notebooks/02_Data_Curation/data/sft"
        print(f"W&B artifact not found ({exc}), using local path: {source_dir}")

    def to_record(entry):
        return {
            "input": PROMPT_TEMPLATE.format(text=entry["input"]),
            "output": entry["output"],
        }

    prepared_files = []
    for filename in ["train.jsonl", "test.jsonl", "validation.jsonl"]:
        src_path = source_dir / filename
        if not src_path.exists():
            print(f"Skipping {src_path} because it does not exist.")
            continue

        with open(src_path, "r", encoding="utf-8") as raw_data_file:
            records = [
                to_record(json.loads(line))
                for line in raw_data_file
                if line.strip()
            ]

        processed_data_path = output_dir / filename
        with open(processed_data_path, "w", encoding="utf-8") as processed_data_file:
            for record in records:
                processed_data_file.write(json.dumps(record, ensure_ascii=False) + "\n")

        prepared_files.append(processed_data_path)
        print(f"Processed and saved: {processed_data_path} ({len(records)} records)")

    assert prepared_files, "No JSONL files processed — please verify the source path or W&B artifact."

    ds_artifact = wandb.Artifact(
        name=PREPARED_DATASET_ARTIFACT,
        type="dataset",
        description="Prompt-formatted SFT splits (train/test/validation) for Llama-3.1-8B-Instruct",
    )
    for fpath in prepared_files:
        ds_artifact.add_file(str(fpath))
    run.log_artifact(ds_artifact, aliases=["latest"]).wait()
    print(f"✅ Dataset artifact '{PREPARED_DATASET_ARTIFACT}' logged to W&B.")

    if own_run:
        run.finish()

    return output_dir

### 3. Training (`nemo-automodel` + W&B Training Telemetry & Model Registry)

Executes Supervised Fine-Tuning via `automodel` with W&B logging attached to the active `WANDB_RUN_ID` (so `train/loss`, `val/loss`, `lr`, `grad_norm`, and step throughput stream directly into W&B) and registers the generated checkpoints as `llama3p1-8b-fraud-lora`.

In [ ]:
def run_training(smoke_test=True, enable_wandb_telemetry=True):
    run_tag = "smoke-test" if smoke_test else "full-sft"
    config_path = Path("Automodel_Finetuning_configs/llama3p1_8b_lora.yaml")
    if not config_path.exists():
        raise FileNotFoundError(f"Config not found: {config_path.resolve()}")

    run = wandb.init(
        project=WANDB_PROJECT,
        job_type="automodel-finetuning",
        name=f"llama3p1-8b-{run_tag}",
        tags=["stage-3-finetuning", "automodel", run_tag],
        config={
            "base_model": "meta-llama/Llama-3.1-8B-Instruct",
            "config_file": str(config_path),
            "smoke_test": smoke_test,
            "wandb_telemetry": enable_wandb_telemetry,
        },
    )

    # Prepare data within the same run so W&B lineage is seamlessly linked
    prepare_data(run=run)

    print("\n=== Starting Training ===")
    run_dir = Path("results") / ("llama3p1_8b_lora_smoke_test" if smoke_test else "llama3p1-8b-lora")
    ckpt_dir = run_dir / "checkpoints"

    env = os.environ.copy()
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    # Share the active W&B run ID with the automodel subprocess so step metrics merge cleanly
    env["WANDB_PROJECT"] = WANDB_PROJECT
    env["WANDB_RUN_ID"] = run.id
    env["WANDB_RESUME"] = "allow"

    base_cmd = [
        "automodel",
        str(config_path),
        "--checkpoint.checkpoint_dir",
        str(ckpt_dir),
    ]
    if smoke_test:
        print("Running smoke test (50 steps)...")
        base_cmd += ["--step_scheduler.max_steps", "50"]
    else:
        print("Running full training...")

    # Enable NeMo AutoModel W&B logger overrides (falls back cleanly if YAML schema differs)
    wandb_overrides = [
        "--wandb.project", WANDB_PROJECT,
        "--wandb.name", f"llama3p1-8b-{run_tag}",
        "--wandb.id", run.id,
    ]

    if enable_wandb_telemetry:
        try:
            subprocess.run(base_cmd + wandb_overrides, check=True, env=env)
        except subprocess.CalledProcessError:
            print("⚠️ CLI wandb overrides not supported by this config schema; retrying with env-based W&B tracking...")
            subprocess.run(base_cmd, check=True, env=env)
    else:
        subprocess.run(base_cmd, check=True, env=env)

    # Register trained model / checkpoints as downstream artifact in W&B
    if ckpt_dir.exists():
        model_artifact = wandb.Artifact(
            name=MODEL_ARTIFACT_NAME,
            type="model",
            description=f"Fine-tuned Llama-3.1-8B-Instruct LoRA ({run_tag})",
            metadata={
                "base_model": "meta-llama/Llama-3.1-8B-Instruct",
                "smoke_test": smoke_test,
                "checkpoint_dir": str(ckpt_dir),
            },
        )
        model_artifact.add_dir(str(ckpt_dir))
        run.log_artifact(model_artifact, aliases=["latest", run_tag]).wait()
        print(f"✅ Model artifact '{MODEL_ARTIFACT_NAME}' logged to W&B.")

    run.finish()
    return ckpt_dir


# Select here: True for smoke test (50 steps), False for full training
SMOKE_TEST = True
ckpt_dir = run_training(smoke_test=SMOKE_TEST, enable_wandb_telemetry=True)

### 4. Comprehensive Evaluation Helper Functions & Runner (with W&B Reporting)

Loads the base model as well as the generated checkpoint (including automatic detection of `LATEST/model/consolidated` or PEFT adapters), computes both **greedy text predictions** and **calibrated first-token fraud probabilities** (`P(fraud)` vs. `P(legitimate)`), and logs:
- **Core & Imbalance Metrics:** Accuracy, Precision, Recall, F1 (`fraud` & `legitimate`), False Positive Rate (FPR), False Negative Rate (FNR), Matthews Correlation Coefficient (MCC), Cohen's Kappa, and Unparsed Output Count.
- **Ranking & Calibration Metrics:** ROC-AUC, PR-AUC (Average Precision), Brier Score, W&B ROC Curve, and W&B Precision-Recall Curve.
- **Diagnostics & Artifacts:** Confusion Matrix, Latency, `model_comparison` table, `prediction_samples` table, `misclassified_samples` error-analysis table, and the JSON/JSONL evaluation report artifact.

In [ ]:
def load_model(checkpoint: str):
    is_local_peft_adapter = (
        Path(checkpoint).is_dir() and (Path(checkpoint) / "adapter_config.json").exists()
    )
    if is_local_peft_adapter:
        from peft import AutoPeftModelForCausalLM

        print(f"Loading PEFT/LoRA adapter from {checkpoint}")
        model = AutoPeftModelForCausalLM.from_pretrained(
            checkpoint, torch_dtype=torch.bfloat16, device_map="cuda"
        )
        tokenizer_source = checkpoint
    else:
        print(f"Loading model {checkpoint}")
        model = AutoModelForCausalLM.from_pretrained(
            checkpoint, torch_dtype=torch.bfloat16, device_map="cuda"
        )
        tokenizer_source = checkpoint

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    model.eval()
    return model, tokenizer


def get_class_token_ids(tokenizer):
    """Resolve single-token IDs for 'fraud' and 'legitimate' (with/without leading space)."""
    fraud_ids = set()
    legit_ids = set()
    for variant in ["fraud", " fraud", "Fraud", " Fraud"]:
        ids = tokenizer.encode(variant, add_special_tokens=False)
        if ids:
            fraud_ids.add(ids[0])
    for variant in ["legitimate", " legitimate", "Legitimate", " Legitimate"]:
        ids = tokenizer.encode(variant, add_special_tokens=False)
        if ids:
            legit_ids.add(ids[0])
    return sorted(fraud_ids), sorted(legit_ids)


def generate_batch_with_probs(model, tokenizer, prompts, fraud_ids, legit_ids, max_new_tokens=16):
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            return_dict_in_generate=True,
            output_scores=True,
        )

    # First generated token logits -> binary probability P(fraud) vs P(legitimate)
    first_step_logits = outputs.scores[0].float()
    fraud_logit = torch.logsumexp(first_step_logits[:, fraud_ids], dim=-1)
    legit_logit = torch.logsumexp(first_step_logits[:, legit_ids], dim=-1)
    two_class_logits = torch.stack([legit_logit, fraud_logit], dim=-1)
    probs = torch.softmax(two_class_logits, dim=-1).cpu().numpy()

    completions = outputs.sequences[:, inputs["input_ids"].shape[1] :]
    decoded = tokenizer.batch_decode(
        completions, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return decoded, probs


def generate_predictions(
    checkpoint: str, input_jsonl: str, output_jsonl: str, limit: int = None, max_batch_size: int = 8
):
    with open(input_jsonl, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f if line.strip()]
    if limit:
        records = records[:limit]

    model, tokenizer = load_model(checkpoint)
    fraud_ids, legit_ids = get_class_token_ids(tokenizer)
    predictions = []
    fraud_probs = []
    batch_timings = []

    for i in range(0, len(records), max_batch_size):
        batch = records[i : i + max_batch_size]
        prompts = [r["input"] for r in batch]
        start = time.perf_counter()
        batch_preds, batch_probs = generate_batch_with_probs(
            model, tokenizer, prompts, fraud_ids, legit_ids
        )
        batch_timings.append({"n": len(batch), "seconds": time.perf_counter() - start})
        predictions.extend(batch_preds)
        fraud_probs.extend(batch_probs[:, 1].tolist())

    # Free VRAM after inference
    del model
    torch.cuda.empty_cache()

    out_path = Path(output_jsonl)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for record, prediction, prob_fraud in zip(records, predictions, fraud_probs):
            out = dict(record)
            out["prediction"] = prediction.strip()
            out["prob_fraud"] = round(float(prob_fraud), 6)
            out["prob_legitimate"] = round(float(1.0 - prob_fraud), 6)
            f.write(json.dumps(out, ensure_ascii=False) + "\n")

    avg_latency = (
        sum(t["seconds"] for t in batch_timings) / len(records) if records else 0.0
    )
    print(f"Predictions written to {output_jsonl} (avg {avg_latency:.3f}s / sample)")
    return avg_latency


def resolve_checkpoint_path(smoke_test=True) -> Path:
    run_dir = Path("results") / (
        "llama3p1_8b_lora_smoke_test" if smoke_test else "llama3p1-8b-lora"
    )
    candidates = [
        run_dir / "checkpoints" / "LATEST" / "model" / "consolidated",
        run_dir / "checkpoints" / "LATEST" / "model",
        run_dir / "checkpoints" / "model" / "consolidated",
        run_dir / "checkpoints" / "model",
    ]
    for cand in candidates:
        if cand.exists():
            return cand

    ckpt_root = run_dir / "checkpoints"
    if ckpt_root.exists():
        epoch_dirs = sorted(ckpt_root.glob("epoch_*"))
        if epoch_dirs:
            return epoch_dirs[-1]

    raise FileNotFoundError(
        f"No valid checkpoint found under {run_dir} — please run training first."
    )


def evaluate_models(smoke_test=True, eval_n=200):
    run_tag = "smoke" if smoke_test else "full"
    run = wandb.init(
        project=WANDB_PROJECT,
        job_type="post-training-evaluation",
        name=f"eval-llama3p1-8b-{run_tag}",
        tags=["stage-4-evaluation", "automodel", run_tag],
        config={"eval_n": eval_n, "smoke_test": smoke_test},
    )

    # Lineage linking to dataset and model artifacts in W&B
    try:
        run.use_artifact(PREPARED_DATASET_ARTIFACT + ":latest")
        run.use_artifact(MODEL_ARTIFACT_NAME + ":latest")
    except Exception as exc:
        print(f"Note on artifact linking: {exc}")

    print("\n=== Starting Comprehensive Evaluation ===")
    BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
    RESULTS_DIR = Path("results")
    checkpoint_path = resolve_checkpoint_path(smoke_test=smoke_test)
    print("Using checkpoint:", checkpoint_path)

    base_output = str(RESULTS_DIR / "eval_base_predictions.jsonl")
    finetuned_output = str(RESULTS_DIR / "eval_finetuned_predictions.jsonl")
    test_jsonl = str(Path("data") / "test.jsonl")

    base_latency = generate_predictions(BASE_MODEL, test_jsonl, base_output, limit=eval_n)
    ft_latency = generate_predictions(
        str(checkpoint_path), test_jsonl, finetuned_output, limit=eval_n
    )

    def normalize(text):
        text = str(text).strip().lower()
        if "fraud" in text:
            return "fraud"
        if "legitimate" in text:
            return "legitimate"
        return "unparsed"

    def load_preds(output_path):
        with open(output_path, "r", encoding="utf-8") as f:
            rows = [json.loads(line) for line in f if line.strip()]
        return (
            [normalize(r["output"]) for r in rows],
            [normalize(r["prediction"]) for r in rows],
            np.array([float(r["prob_fraud"]) for r in rows]),
            rows,
        )

    true_labels, base_preds, base_probs, _ = load_preds(base_output)
    _, finetuned_preds, ft_probs, ft_rows = load_preds(finetuned_output)

    labels = ["fraud", "legitimate"]
    cm_labels = ["fraud", "legitimate", "unparsed"]
    label_to_idx = {name: idx for idx, name in enumerate(cm_labels)}
    y_true_bin = np.array([1 if t == "fraud" else 0 for t in true_labels])
    summary_rows = []

    for key, label, preds, probs_fraud, latency in [
        ("base", "Base model (zero-shot)", base_preds, base_probs, base_latency),
        ("finetuned", "Fine-tuned", finetuned_preds, ft_probs, ft_latency),
    ]:
        print(f"\n=== {label} ===")
        print(classification_report(true_labels, preds, labels=labels, zero_division=0))

        cm = confusion_matrix(true_labels, preds, labels=labels)
        print("Confusion matrix (rows=true, cols=predicted, order=[fraud, legitimate]):")
        print(cm)

        # Extract TP, FN, FP, TN from [[TP, FN], [FP, TN]]
        tp, fn = int(cm[0, 0]), int(cm[0, 1])
        fp, tn = int(cm[1, 0]), int(cm[1, 1])
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

        acc = accuracy_score(true_labels, preds)
        prec, rec, f1, _ = precision_recall_fscore_support(
            true_labels, preds, labels=labels, average=None, zero_division=0
        )
        macro_f1 = float(np.mean(f1))
        mcc = float(matthews_corrcoef(true_labels, preds))
        kappa = float(cohen_kappa_score(true_labels, preds))
        unparsed = sum(1 for p in preds if p == "unparsed")

        # Probability-based ranking & calibration metrics
        has_both_classes = len(np.unique(y_true_bin)) == 2
        roc_auc = float(roc_auc_score(y_true_bin, probs_fraud)) if has_both_classes else 0.0
        pr_auc = float(average_precision_score(y_true_bin, probs_fraud)) if has_both_classes else 0.0
        brier = float(brier_score_loss(y_true_bin, probs_fraud))

        print(
            f"MCC: {mcc:.4f} | Kappa: {kappa:.4f} | FPR: {fpr:.4f} | FNR: {fnr:.4f} | "
            f"ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f} | Brier: {brier:.4f}"
        )
        if unparsed:
            print(f"({unparsed} predictions could not be parsed as fraud/legitimate)")

        y_true_idx = [label_to_idx[t] for t in true_labels]
        preds_idx = [label_to_idx[p] for p in preds]
        y_probas_2col = np.column_stack([1.0 - probs_fraud, probs_fraud])

        log_payload = {
            f"{key}/accuracy": acc,
            f"{key}/macro_f1": macro_f1,
            f"{key}/mcc": mcc,
            f"{key}/cohen_kappa": kappa,
            f"{key}/fraud_precision": prec[0],
            f"{key}/fraud_recall": rec[0],
            f"{key}/fraud_f1": f1[0],
            f"{key}/legitimate_precision": prec[1],
            f"{key}/legitimate_recall": rec[1],
            f"{key}/legitimate_f1": f1[1],
            f"{key}/false_positive_rate": fpr,
            f"{key}/false_negative_rate": fnr,
            f"{key}/roc_auc": roc_auc,
            f"{key}/pr_auc": pr_auc,
            f"{key}/brier_score": brier,
            f"{key}/unparsed_count": unparsed,
            f"{key}/avg_latency_sec": latency,
            f"{key}/confusion_matrix": wandb.plot.confusion_matrix(
                probs=None,
                y_true=y_true_idx,
                preds=preds_idx,
                class_names=cm_labels,
            ),
        }

        if has_both_classes:
            log_payload[f"{key}/roc_curve"] = wandb.plot.roc_curve(
                y_true_bin, y_probas_2col, labels=["legitimate", "fraud"]
            )
            log_payload[f"{key}/pr_curve"] = wandb.plot.pr_curve(
                y_true_bin, y_probas_2col, labels=["legitimate", "fraud"]
            )

        run.log(log_payload)

        summary_rows.append([
            label,
            acc,
            macro_f1,
            mcc,
            prec[0],
            rec[0],
            f1[0],
            f1[1],
            fpr,
            fnr,
            roc_auc,
            pr_auc,
            brier,
            unparsed,
            latency,
        ])

    # Log uplift deltas (Fine-tuned minus Base) for quick executive tracking
    run.log({
        "delta/accuracy": summary_rows[1][1] - summary_rows[0][1],
        "delta/mcc": summary_rows[1][3] - summary_rows[0][3],
        "delta/fraud_f1": summary_rows[1][6] - summary_rows[0][6],
        "delta/roc_auc": summary_rows[1][10] - summary_rows[0][10],
        "delta/pr_auc": summary_rows[1][11] - summary_rows[0][11],
        "delta/fpr_reduction": summary_rows[0][8] - summary_rows[1][8],
    })

    # Log comparison table & sample predictions to W&B
    columns = [
        "model",
        "accuracy",
        "macro_f1",
        "mcc",
        "fraud_precision",
        "fraud_recall",
        "fraud_f1",
        "legitimate_f1",
        "fpr",
        "fnr",
        "roc_auc",
        "pr_auc",
        "brier_score",
        "unparsed",
        "avg_latency_sec",
    ]
    run.log({"model_comparison": wandb.Table(columns=columns, data=summary_rows)})

    sample_table = wandb.Table(
        columns=[
            "transcript_prompt",
            "ground_truth",
            "base_pred",
            "base_prob_fraud",
            "finetuned_pred",
            "finetuned_prob_fraud",
        ],
        data=[
            [
                r["input"][:300],
                r["output"],
                b_p,
                round(float(b_prob), 4),
                f_p,
                round(float(f_prob), 4),
            ]
            for r, b_p, b_prob, f_p, f_prob in list(
                zip(ft_rows, base_preds, base_probs, finetuned_preds, ft_probs)
            )[:25]
        ],
    )
    run.log({"prediction_samples": sample_table})

    # Filtered error-analysis table: only rows where fine-tuned model failed or remained unparsed
    misclassified_rows = [
        [
            r["input"][:400],
            gt,
            f_p,
            round(float(f_prob), 4),
            "False Positive" if (gt == "legitimate" and f_p == "fraud") else (
                "False Negative" if (gt == "fraud" and f_p == "legitimate") else "Unparsed"
            ),
        ]
        for r, gt, f_p, f_prob in zip(ft_rows, true_labels, finetuned_preds, ft_probs)
        if gt != f_p
    ]
    misclassified_table = wandb.Table(
        columns=[
            "transcript_prompt",
            "ground_truth",
            "finetuned_pred",
            "finetuned_prob_fraud",
            "error_type",
        ],
        data=misclassified_rows,
    )
    run.log({
        "misclassified_samples": misclassified_table,
        "finetuned/error_count": len(misclassified_rows),
    })

    # Save report JSON & prediction files as W&B artifact
    report_path = RESULTS_DIR / "post_training_metrics_summary.json"
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "base": dict(zip(columns, summary_rows[0])),
                "finetuned": dict(zip(columns, summary_rows[1])),
            },
            f,
            indent=2,
        )

    report_art = wandb.Artifact(
        name=EVAL_REPORT_ARTIFACT,
        type="evaluation-report",
        description="Comprehensive comparison metrics (Base vs. Fine-tuned) for Llama-3.1-8B-Instruct Fraud Detection",
    )
    report_art.add_file(str(report_path))
    report_art.add_file(base_output)
    report_art.add_file(finetuned_output)
    run.log_artifact(report_art, aliases=["latest", run_tag]).wait()

    run.finish()
    print("✅ Comprehensive evaluation and W&B report completed.")


# Run evaluation
evaluate_models(smoke_test=SMOKE_TEST, eval_n=200)

### 5. Executive Summary

#### 🎯 What This Pipeline Does
* **Goal:** Automatically classify incoming phone call transcripts as **fraudulent** (`fraud`) or **legitimate** (`legitimate`) while keeping false alarms and missed fraud cases low.
* **Model & Customization:** Adapts **meta-llama/Llama-3.1-8B-Instruct** via parameter-efficient LoRA fine-tuning (`llama3p1_8b_lora.yaml`) on curated call transcripts (`curated-sft-dataset:latest` → `prepared-sft-dataset`).
* **End-to-End Traceability:** Every dataset split, training run (`llama3p1-8b-fraud-lora`), and evaluation report (`post-training-evaluation-report`) is versioned in Weights & Biases (`nemo-fraud-detection`).

#### 📈 How Success Is Measured (Base Model vs. Fine-Tuned)
* **Overall & Imbalance-Robust Quality:** Accuracy, Macro-F1, Matthews Correlation Coefficient (`mcc`), and Cohen's Kappa (`cohen_kappa`) — tracked as uplift deltas (`delta/accuracy`, `delta/mcc`, `delta/fraud_f1`).
* **Fraud Detection Trade-Offs:** `fraud_precision`, `fraud_recall`, `fraud_f1`, False Positive Rate (`false_positive_rate`, tracked as `delta/fpr_reduction`), and False Negative Rate (`false_negative_rate`).
* **Probability Ranking & Calibration:** `roc_auc`, `pr_auc`, and `brier_score`, paired with `model_comparison`, `prediction_samples`, and `misclassified_samples` error tables.

#### 🛡️ Guardrail & Compliance Coverage (`rails.co`)
* **PII Protection (`is_pii_present`):** Checks transcripts for personal identifiable information before approval.
* **Prohibited Content Blocking (`express prohibited content`):** Automatically refuses transcripts containing explicit credential-theft or unauthorized-transfer instructions (such as asking for passwords, PINs, or direct account transfers) and only approves compliant transcripts (`bot approve`).